
                # Instagram + Twitter/X Scraper for Google Colab

                Notebook ini disiapkan agar owner bisa mencoba langsung di Google Colab.

                Catatan penting:
                - Tidak ada akun, cookie, atau sesi login yang disertakan di notebook ini.
                - Jika scraping Instagram membutuhkan login, user mengisi sendiri username dan password pada runtime Colab.
                - Output akhir disimpan dalam format CSV dan JSON.
                - Untuk Twitter/X, jalur default yang paling aman adalah input manual atau browser-assisted, bukan bergantung penuh pada public search. Mode snscrape hanya eksperimen.


In [ ]:
!pip install -q instaloader pandas



                ## 1. Isi parameter target

                Ubah username target, tanggal, dan keyword sesuai kebutuhan.
                Untuk Instagram login, biarkan kosong jika mau coba tanpa login terlebih dahulu.
                Untuk Twitter/X, default `TWITTER_MODE = "manual"` agar notebook tetap stabil.


In [ ]:

    from pathlib import Path
    from datetime import datetime, timezone, timedelta
    import csv
    import getpass
    import json
    import re

    OUTPUT_ROOT = Path("output")
    OUTPUT_ROOT.mkdir(exist_ok=True)

    INSTAGRAM_USERNAME = "isi_username_target"
    INSTAGRAM_LOGIN_USERNAME = ""
    INSTAGRAM_LOGIN_PASSWORD = ""
    INSTAGRAM_START_DATE = "2026-06-01"
    INSTAGRAM_END_DATE = "2026-06-30"
    INSTAGRAM_KEYWORDS = [
        "spmb",
        "pcmb",
        "pendaftaran",
        "pengumuman",
        "hasil seleksi",
        "jalur",
        "zonasi",
        "afirmasi",
        "domisili",
    ]

    TWITTER_START_DATE = "2026-06-01"
    TWITTER_END_DATE = "2026-06-30"
    TWITTER_LIMIT = 100
    TWITTER_MODE = "manual"
    TWITTER_MANUAL_ROWS = [
        {
            "source_query": "SPMB Jabar",
            "tweet_url": "https://x.com/contoh/status/1234567890",
            "created_at_utc": "2026-06-21T03:04:00+00:00",
            "username": "contoh_user",
            "display_name": "Contoh User",
            "content": "Contoh isi tweet hasil copy dari browser atau input manual.",
            "reply_count": 0,
            "retweet_count": 0,
            "like_count": 0,
            "quote_count": 0,
            "lang": "id",
            "is_reply": False,
            "in_reply_to_tweet_id": "",
            "hashtags": "SPMB2026",
        }
    ]
    TWITTER_HASHTAGS = ["#SPMB2026", "#PCMB2026", "#SPMBJabar"]
    TWITTER_PHRASES = ["SPMB Jabar", "PCMB Jabar", "system error", "pengumuman molor"]
    TWITTER_BOOLEAN_SNIPPETS = [
        "SPMB (error OR down OR gagal OR lambat)",
        "PCMB (pengumuman OR hasil OR \"tidak bisa akses\")",
    ]



                ## 2. Jalankan scraping Instagram

                Sel ini akan menyimpan hasil ke folder `output/instagram`.


In [ ]:

    import instaloader
    from instaloader.exceptions import InstaloaderException, ProfileNotExistsException

    TEXT_RE = re.compile(r"\w", re.UNICODE)

    def normalize_text(value: str) -> str:
        return re.sub(r"\s+", " ", (value or "")).strip()

    def has_substantive_text(value: str) -> bool:
        return bool(TEXT_RE.search(value))

    def contains_keyword(value: str, keywords: list[str]) -> bool:
        lowered = value.casefold()
        return any(keyword.casefold() in lowered for keyword in keywords)

    def parse_date(value: str, end_of_day: bool = False) -> datetime:
        dt = datetime.strptime(value, "%Y-%m-%d")
        if end_of_day:
            dt = dt.replace(hour=23, minute=59, second=59)
        return dt.replace(tzinfo=timezone.utc)

    def within_range(dt: datetime, start_dt: datetime, end_dt: datetime) -> bool:
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return start_dt <= dt <= end_dt

    def save_rows(rows: list[dict], output_dir: Path, stem: str) -> None:
        output_dir.mkdir(parents=True, exist_ok=True)
        json_path = output_dir / f"{stem}.json"
        csv_path = output_dir / f"{stem}.csv"
        with json_path.open("w", encoding="utf-8") as handle:
            json.dump(rows, handle, ensure_ascii=False, indent=2)
        with csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()) if rows else [])
            if rows:
                writer.writeheader()
                writer.writerows(rows)

    start_dt = parse_date(INSTAGRAM_START_DATE)
    end_dt = parse_date(INSTAGRAM_END_DATE, end_of_day=True)
    loader = instaloader.Instaloader(
        download_pictures=False,
        download_videos=False,
        download_video_thumbnails=False,
        download_geotags=False,
        download_comments=False,
        save_metadata=False,
        compress_json=False,
        quiet=False,
    )

    if INSTAGRAM_LOGIN_USERNAME and not INSTAGRAM_LOGIN_PASSWORD:
        INSTAGRAM_LOGIN_PASSWORD = getpass.getpass("Masukkan password Instagram untuk sesi ini: ")

    if INSTAGRAM_LOGIN_USERNAME and INSTAGRAM_LOGIN_PASSWORD:
        loader.login(INSTAGRAM_LOGIN_USERNAME, INSTAGRAM_LOGIN_PASSWORD)

    rows = []
    seen_text = set()

    try:
        profile = instaloader.Profile.from_username(loader.context, INSTAGRAM_USERNAME)
        for post in profile.get_posts():
            try:
                comments_iterable = post.get_comments()
            except InstaloaderException as exc:
                print(f"Skipping post {post.shortcode}: {exc}")
                continue

            for comment in comments_iterable:
                text = normalize_text(comment.text)
                if not within_range(comment.created_at_utc, start_dt, end_dt):
                    continue
                if not has_substantive_text(text):
                    continue
                if not contains_keyword(text, INSTAGRAM_KEYWORDS):
                    continue
                dedupe_key = text.casefold()
                if dedupe_key in seen_text:
                    continue
                seen_text.add(dedupe_key)
                rows.append(
                    {
                        "platform": "instagram",
                        "profile_username": post.owner_username,
                        "post_shortcode": post.shortcode,
                        "post_url": f"https://www.instagram.com/p/{post.shortcode}/",
                        "post_caption": normalize_text(post.caption or ""),
                        "post_date_utc": post.date_utc.isoformat(),
                        "comment_id": str(comment.id),
                        "comment_text": text,
                        "comment_created_at_utc": comment.created_at_utc.isoformat(),
                        "comment_owner_username": getattr(comment.owner, "username", ""),
                        "comment_owner_id": getattr(comment.owner, "userid", ""),
                        "like_count": getattr(comment, "likes_count", None),
                        "is_answer": False,
                        "parent_comment_id": "",
                    }
                )
    except ProfileNotExistsException:
        print("Username Instagram target tidak ditemukan.")
    except InstaloaderException as exc:
        print(f"Request Instagram gagal: {exc}")

    instagram_output = OUTPUT_ROOT / "instagram"
    stem = f"{INSTAGRAM_USERNAME}_{INSTAGRAM_START_DATE}_to_{INSTAGRAM_END_DATE}"
    save_rows(rows, instagram_output, stem)
    print(f"Instagram rows tersimpan: {len(rows)}")



                ## 3. Jalankan scraping Twitter/X

                Secara default, sel ini memakai input manual/browser-assisted yang kamu isi di `TWITTER_MANUAL_ROWS`.
                Jika ingin mencoba public search lama, ubah `TWITTER_MODE` ke mode eksperimen.


In [ ]:

    import importlib
    import importlib.machinery

    def _compat_find_module(self, fullname, path=None):
        spec = self.find_spec(fullname, path)
        return spec.loader if spec else None

    finder_classes = [importlib.machinery.FileFinder]
    bootstrap_external = getattr(importlib, "_bootstrap_external", None)
    if bootstrap_external and getattr(bootstrap_external, "FileFinder", None):
        finder_classes.append(bootstrap_external.FileFinder)

    for finder_class in finder_classes:
        if not hasattr(finder_class, "find_module"):
            finder_class.find_module = _compat_find_module

    import json
    try:
        import snscrape.modules.twitter as sntwitter
    except Exception:
        sntwitter = None

    URL_ONLY_RE = re.compile(r"^(https?://\S+|www\.\S+)+$", re.IGNORECASE)

    def build_queries(start_date: str, end_date: str, hashtags: list[str], phrases: list[str], snippets: list[str]) -> list[str]:
        until_date = (datetime.strptime(end_date, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
        queries = []
        if hashtags:
            queries.append(f"({' OR '.join(hashtags)}) since:{start_date} until:{until_date}")
        if phrases:
            quoted = " OR ".join(f'\"{phrase}\"' for phrase in phrases)
            queries.append(f"({quoted}) since:{start_date} until:{until_date}")
        for snippet in snippets:
            queries.append(f"({snippet}) since:{start_date} until:{until_date}")
        return queries

    def is_link_only(value: str) -> bool:
        return bool(URL_ONLY_RE.fullmatch(value.replace(" ", "")))

    def save_rows(rows: list[dict], output_dir: Path, stem: str) -> None:
        output_dir.mkdir(parents=True, exist_ok=True)
        json_path = output_dir / f"{stem}.json"
        csv_path = output_dir / f"{stem}.csv"
        with json_path.open("w", encoding="utf-8") as handle:
            json.dump(rows, handle, ensure_ascii=False, indent=2)
        with csv_path.open("w", newline="", encoding="utf-8-sig") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()) if rows else [])
            if rows:
                writer.writeheader()
                writer.writerows(rows)

    def parse_iso(value: str):
        if not value:
            return None
        cleaned = value.replace("Z", "+00:00")
        try:
            dt = datetime.fromisoformat(cleaned)
        except ValueError:
            return None
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)

    def within_range_text(value: str) -> bool:
        dt = parse_iso(value)
        if dt is None:
            return True
        start_dt = datetime.strptime(TWITTER_START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        end_dt = datetime.strptime(TWITTER_END_DATE, "%Y-%m-%d").replace(hour=23, minute=59, second=59, tzinfo=timezone.utc)
        return start_dt <= dt <= end_dt

    def normalize_manual_row(row: dict) -> dict:
        return {
            "platform": "twitter",
            "source_mode": "manual_import",
            "source_query": row.get("source_query", ""),
            "tweet_id": row.get("tweet_id", ""),
            "tweet_url": row.get("tweet_url", row.get("url", "")),
            "created_at_utc": row.get("created_at_utc", ""),
            "username": row.get("username", ""),
            "display_name": row.get("display_name", ""),
            "user_id": row.get("user_id", ""),
            "content": normalize_text(row.get("content", row.get("text", ""))),
            "reply_count": row.get("reply_count", 0),
            "retweet_count": row.get("retweet_count", 0),
            "like_count": row.get("like_count", 0),
            "quote_count": row.get("quote_count", 0),
            "lang": row.get("lang", ""),
            "is_reply": bool(row.get("is_reply", False)),
            "in_reply_to_tweet_id": row.get("in_reply_to_tweet_id", ""),
            "hashtags": row.get("hashtags", ""),
        }

    rows = []
    seen_ids = set()
    seen_text = set()

    if TWITTER_MODE == "manual":
        for raw_row in TWITTER_MANUAL_ROWS:
            row = normalize_manual_row(raw_row)
            content = row["content"]
            if not has_substantive_text(content):
                continue
            if is_link_only(content):
                continue
            if not within_range_text(row["created_at_utc"]):
                continue
            if content.casefold() in seen_text:
                continue
            tweet_id = str(row["tweet_id"]).strip()
            if tweet_id and tweet_id in seen_ids:
                continue
            if tweet_id:
                seen_ids.add(tweet_id)
            seen_text.add(content.casefold())
            rows.append(row)
            if TWITTER_LIMIT and len(rows) >= TWITTER_LIMIT:
                break
        print("Twitter/X diproses dengan mode manual/browser-assisted.")
    else:
        if sntwitter is None:
            print("snscrape tidak tersedia. Ganti TWITTER_MODE ke 'manual'.")
        else:
            queries = build_queries(
                TWITTER_START_DATE,
                TWITTER_END_DATE,
                TWITTER_HASHTAGS,
                TWITTER_PHRASES,
                TWITTER_BOOLEAN_SNIPPETS,
            )
            for query in queries:
                try:
                    for tweet in sntwitter.TwitterSearchScraper(query).get_items():
                        content = normalize_text(tweet.rawContent)
                        if not has_substantive_text(content):
                            continue
                        if is_link_only(content):
                            continue
                        if content.casefold() in seen_text:
                            continue
                        tweet_id = str(tweet.id)
                        if tweet_id in seen_ids:
                            continue

                        seen_text.add(content.casefold())
                        seen_ids.add(tweet_id)
                        rows.append(
                            {
                                "platform": "twitter",
                                "source_mode": "snscrape_experimental",
                                "source_query": query,
                                "tweet_id": tweet_id,
                                "tweet_url": tweet.url,
                                "created_at_utc": tweet.date.isoformat(),
                                "username": getattr(tweet.user, "username", ""),
                                "display_name": getattr(tweet.user, "displayname", ""),
                                "user_id": getattr(tweet.user, "id", ""),
                                "content": content,
                                "reply_count": tweet.replyCount,
                                "retweet_count": tweet.retweetCount,
                                "like_count": tweet.likeCount,
                                "quote_count": tweet.quoteCount,
                                "lang": tweet.lang,
                                "is_reply": bool(tweet.inReplyToTweetId),
                                "in_reply_to_tweet_id": tweet.inReplyToTweetId or "",
                                "hashtags": ",".join(tweet.hashtags or []),
                            }
                        )
                        if TWITTER_LIMIT and len(rows) >= TWITTER_LIMIT:
                            break
                except Exception as exc:
                    print(f"Query gagal dan dilewati: {query} -> {exc}")
                if TWITTER_LIMIT and len(rows) >= TWITTER_LIMIT:
                    break

    twitter_output = OUTPUT_ROOT / "twitter"
    stem = f"twitter_{TWITTER_START_DATE}_to_{TWITTER_END_DATE}"
    save_rows(rows, twitter_output, stem)
    print(f"Twitter/X rows tersimpan: {len(rows)}")



                ## 4. Preview hasil

                Gunakan sel ini untuk melihat contoh beberapa baris output.


In [ ]:

    import pandas as pd

    instagram_files = sorted((OUTPUT_ROOT / "instagram").glob("*.csv"))
    twitter_files = sorted((OUTPUT_ROOT / "twitter").glob("*.csv"))

    if instagram_files:
        display(pd.read_csv(instagram_files[-1]).head())
    else:
        print("Belum ada output Instagram.")

    if twitter_files:
        display(pd.read_csv(twitter_files[-1]).head())
    else:
        print("Belum ada output Twitter/X.")
